# 4.2  From Whole Documents to Relevant Context: Introducing RAG

Retrieval-Augmented Generation (RAG) exists for a simple reason: context is limited, and relevance matters.

Instead of pasting entire documents into a prompt, RAG adds a retrieval step. When a user asks a question, the system searches your corpus and pulls back only the relevant portions. Those become the model's context.

Simple in theory.

But what counts as a "portion"?

* A chapter?
* A page?
* A paragraph?
* A fixed number of tokens?

It depends and that choice is architectural. Too large, and you waste context. Too small, and you fragment meaning. The way you define these units directly shapes retrieval quality.

RAG isn't just "search before you generate." It's a set of design decisions about how knowledge is segmented and served to the model.

And whatever you choose as your unit of meaning — that's what your system gets to think with.


### 4.2.1  Install Dependencies

In order to use RAG, we need three things:


* ChromaDB: a lightweight vector store that runs in-process (no infrastructure needed)
* sentence-transformers: for turning text into embeddings (numerical representations of meaning)
* tiktoken: for measuring token counts so we can verify our chunks and prompts stay within budget

>Note: ChromaDB was chosen deliberately. It runs inside this notebook with zero external services. This keeps the focus on concepts, not infrastructure.



In [4]:
! pip install chromadb sentence-transformers tiktoken -q

After installing the prerequisite libraries (`chromadb`, `sentence-transformers`, and `tiktoken`), the next cell begins preparing the runtime environment. 

### 4.2.2 Preparing the Runtime Environment (Jupyter)

The line `os.environ["TQDM_DISABLE"] = "1"` disables tqdm progress bars, which are often triggered by libraries like sentence-transformers during embedding generation. In Jupyter environments, those progress bars can occasionally cause rendering glitches or thread lock issues, so this is a small stability safeguard. It doesn’t change any retrieval logic, it simply ensures the notebook runs cleanly during the workshop. 

In [5]:
# Environment fix: disable tqdm progress bars to avoid Jupyter thread lock issues
import os
os.environ["TQDM_DISABLE"] = "1"

import requests
import json

### 4.2.3 Test API Connection
Before we start embedding and retrieving, we need to confirm that the notebook can reach the MaaS endpoint using the same credentials from Section 2. 

This is a quick connectivity check. it sends a minimal request to the API and verifies we get a valid response. If this cell fails, nothing downstream will work, so it's worth catching the issue here rather than debugging it later inside a retrieval call.


In [6]:
import sys
sys.path.insert(0, "..")
from config import API_KEY as key, ENDPOINT_BASE as endpoint_base
url_chat = f"{endpoint_base}/chat/completions"


# Quick connection test
headers = {"Authorization": f"Bearer {key}", "Content-Type": "application/json"}
test_data = {"model": "granite-3-2-8b-instruct", "messages": [{"role": "user", "content": "ping"}], "max_tokens": 5}
resp = requests.post(url_chat, headers=headers, json=test_data)
if resp.status_code == 200:
    print("✅ API connection working")
else:
    print(f"❌ API connection failed: {resp.status_code}")

✅ API connection working


If you see `✅ API connection working` you are good to proceed.

### 4.2.4 Utility: Batched Vector Store Loading
ChromaDB has a maximum batch size for adding documents to a collection. When we start loading chunks, we may exceed that limit depending on document size and chunking strategy. 

This helper function handles that automatically for us. It splits the chunks into batches and loads them sequentially so we don't have to think about ChromaDB's internal limits during the rest of the lab. You won't call this function directly in the next few cells, but it will be used once we're ready to load our chunks into the vector store.


In [5]:
def add_to_collection(collection, chunks, embeddings, batch_size=5000):
    for i in range(0, len(chunks), batch_size):
        end = min(i + batch_size, len(chunks))
        collection.add(
            documents=chunks[i:end],
            embeddings=embeddings[i:end].tolist() if hasattr(embeddings, 'tolist') else embeddings[i:end],
            ids=[f"chunk_{j}" for j in range(i, end)]
        )
    print(f"✅ Loaded {collection.count()} chunks into the vector store")

## 4.3 Chunking Is a Design Decision, Not a Default

In 4.2, we said RAG retrieves the “right portions” of your data. This is where that idea gets real.
If retrieval is about selecting the right context, chunking defines what “right” can even mean.
When you split documents into chunks, you’re not just preparing text for embeddings. You’re making architectural decisions about:

* How much context travels together
* Which semantic boundaries are preserved
* What the retriever is even capable of returning
* What the model can reason over in a single pass

RAG can only retrieve what exists as a retrievable unit. If your chunks are incoherent, fragmented, or arbitrarily sliced, “relevance” becomes constrained by those boundaries.

![Where Chunking Fits](images/ragchart.png)



A chunk is not just a slice of text. It’s a unit of reasoning.
And that means chunking isn’t a backend implementation detail. It’s part of the product design. It shapes answer quality, system behavior, and ultimately user trust.

Field takeaway: 
>Chunking strategy is a customer conversation, not a backend detail.


### 4.3.1 Load the Docling Output

We pick up where Section 3 left off. The markdown file is the **contract** between ingestion and retrieval.


In [6]:
with open("Basic-Fantasy-RPG-Rules-r142.md", "r", encoding="utf-8") as f:
    full_text = f.read()

print(f"Document length: {len(full_text):,} characters")
print(f"\nFirst 500 characters:\n{full_text[:500]}...")

Document length: 908,803 characters

First 500 characters:
<!-- image -->

Copyright © 2006-2025 Chris Gonnerman All Rights Reserved.  See next page for license information. www.basicfantasy.org

Dedicated to Gary Gygax, Dave Arneson, Tom Moldvay, David Cook, and Stephen Marsh and to my daughter Taylor, my first and best inspiration

## Basic Fantasy Role-Playing Game

4th Edition, Release 142

Copyright © 2006-2025 Chris Gonnerman - All Rights Reserved

<!-- image -->

All textual materials in this document, as well as all maps, floorplans, diagrams, c...


You’ll see that the document is over 908,000 characters along with the first 500 characters shown in the response.

### 4.3.2 Naive Chunking: Fixed Size

We start with the simplest possible approach: split the document into fixed-size chunks with overlap. This is the most common starting point.

Key parameters:
* `chunk_size`: How many characters per chunk. Larger = more context per chunk, but less precision. Smaller = more precise retrieval, but risks cutting mid-thought.
* `overlap`: Characters shared between adjacent chunks. Prevents information from being lost at chunk boundaries. If a rule spans two chunks, overlap ensures at least one chunk has the complete rule.

There are more sophisticated approaches (section-aware splitting, semantic chunking), but this demonstrates the core tradeoff clearly.

We can adjust our strategy later based on the desired results.



In [8]:
def chunk_text_naive(text, chunk_size=1000, overlap=200):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

chunks_naive = chunk_text_naive(full_text, chunk_size=1000, overlap=200)

print(f"Total chunks created: {len(chunks_naive)}")
print(f"Average chunk length: {sum(len(c) for c in chunks_naive) / len(chunks_naive):.0f} characters")

Total chunks created: 1137
Average chunk length: 999 characters


### 4.3.3 The 1000 / 200 Starting Point
A common baseline is `chunk_size = 1000` with `overlap = 200`. A 1000-character chunk is typically large enough to preserve a coherent thought, often a few paragraphs, without becoming so large that retrieval loses precision. The 200-character overlap acts as insurance: if a key idea starts near the end of one chunk, that buffer ensures it reappears at the beginning of the next, reducing boundary damage without dramatically increasing index size.

Is 1000/200 universally correct? No. It’s a pragmatic starting point: large enough for coherence, small enough for precision, and with enough overlap to protect meaning. From there, you adjust: shrink chunks if retrieval is noisy, grow them if answers feel fragmented, increase overlap if boundary issues surface. The tradeoff remains constant: balancing context cohesion against retrieval precision.


In [9]:
def chunk_text(text, chunk_size=500, overlap=100):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap  # Step forward, minus overlap
    return chunks

# Chunk with our default parameters
chunks = chunk_text(full_text, chunk_size=1000, overlap=200)

print(f"Total chunks created: {len(chunks)}")
print(f"Average chunk length: {sum(len(c) for c in chunks) / len(chunks):.0f} characters")


Total chunks created: 1137
Average chunk length: 999 characters


### 4.3.4 Inspect the Naive Chunks

Before we embed anything, **look at what we produced.** Where did the splits land? Did we cut in the middle of a word? A rule? A table?


In [10]:
# Look at chunk boundaries — check for mid-word cuts
print("--- End of Chunk 0 ---")
print(f"...{chunks_naive[0][-80:]}")
print(f"\n--- Start of Chunk 1 ---")
print(f"{chunks_naive[1][:80]}...")

print(f"\n{'=' * 60}")
print("Notice anything? Does the text cut cleanly or mid-word?")
print(f"{'=' * 60}")

--- End of Chunk 0 ---
... so.

The full license text can be viewed at:

https://creativecommons.org/licen

--- Start of Chunk 1 ---
ribute this work as is without permission of the original artists; you must remo...

Notice anything? Does the text cut cleanly or mid-word?


The text is cut in the middle of a word.

When your chunker slices a word in half, it feels like a cosmetic issue. A little rough edge in preprocessing. No big deal, right?

It is a big deal. Because nothing in an LLM pipeline is isolated. Every step hands its output to the next one, and small imperfections compound.

Let’s walk it forward.

That chunk you just created? It gets embedded — transformed into a vector, a dense list of numbers meant to represent meaning. The embedding model sees the text:

>“ribute this work as is without permission”

Now it has a problem. “ribute” isn’t a word. The model doesn’t throw an exception. It doesn’t light up a warning dashboard. It just does what neural networks do: it guesses.

It leans on surrounding context. It approximates. It smooths over the noise.

But the signal is corrupted.

The vector it produces isn’t catastrophic: it’s just slightly off. And “slightly off” is the most dangerous kind of failure in AI systems.

Now multiply that across hundreds or thousands of chunks. Some splits are harmless. Breaking between sentences? Fine. Cutting through whitespace? No one cares.

But sometimes the knife lands on something important:

* Domain-specific terminology
* Proper nouns
* Multi-word concepts

If `“saving throw”` becomes `“saving thr”` and `“ow,”` neither chunk faithfully represents the concept anymore. When a user asks, “How do saving throws work?” your retrieval layer is now comparing their clean query against vectors that never fully captured the term in the first place.

The system isn’t broken. It’s degraded. And that’s the insidious part.

Embedding models are remarkably robust to noise. They will tolerate abuse. They will compensate. They will keep producing plausible outputs. So you don’t see a red flag. You see something worse:

Slightly worse retrieval. Slightly less relevant context. Slightly muddier answers

The AI just feels a little, for lack of a better word, dumber.

And because the failure shows up at the answer layer, we blame the model. We tweak prompts. We swap embedding providers. We adjust the temperature setting on the LLM.

Meanwhile, the real problem is upstream. Quietly introduced during preprocessing.

Chunking isn’t a throwaway step. It’s not clerical work you hand off to a utility function and forget.

It’s a design decision and a critical consideration.

It defines the atomic units of meaning in your system. It shapes how information is represented, retrieved, and reasoned over. It sets the quality ceiling long before a user ever types a question.

Garbage in isn’t always garbage out. Sometimes it’s just slightly worse out. And in production systems, that’s enough to matter. 


### 4.3.5 Respecting Structure: Markdown-Aware Chunking

The naive chunker treated our document as a stream of characters — a long string to be sliced at fixed intervals, regardless of what it was cutting through. But the output from Docling isn't a raw character stream. I

t's Markdown. It has structure: headings, paragraphs, list items, table rows. We spent real effort in Section 3 preserving that structure. A chunking strategy that ignores it is throwing away work we already did.

The simplest improvement doesn't require a new library or a different algorithm. We just stop cutting in the middle of words. Same chunk size, same overlap — but before we make the cut, we back up to the nearest whitespace boundary. It's a small change with a disproportionate effect.

This won't solve every chunking problem. It doesn't respect section boundaries, it doesn't keep tables intact, and it doesn't understand that a heading and the paragraph beneath it belong together. But it eliminates an entire class of silent failures — the ones where embeddings are built on fragments that aren't even valid words.

Think of it as the minimum standard: if your chunker is producing tokens that a human couldn't read aloud, something upstream needs to change before you blame anything downstream.

Let’s change our code to match this approach.



In [11]:
def chunk_text_markdown(text, chunk_size=1000, overlap=200):
    """Split markdown into chunks that respect structural boundaries.
    
    Splits on headings and paragraph breaks first, then accumulates
    blocks into chunks within the size budget. Falls back to word-boundary
    splitting only when a single block exceeds chunk_size.
    """
    import re
    
    # Split on markdown headings (keep the heading with the text that follows)
    # and on double newlines (paragraph boundaries)
    sections = re.split(r'(?=\n#{1,6} )', text)
    
    # Further split each section on paragraph breaks (double newline)
    blocks = []
    for section in sections:
        paragraphs = re.split(r'\n{2,}', section)
        for p in paragraphs:
            stripped = p.strip()
            if stripped:
                blocks.append(stripped)
    
    # Accumulate blocks into chunks
    chunks = []
    current_chunk = ""
    
    for block in blocks:
        # If a single block exceeds chunk_size, split it at word boundaries
        if len(block) > chunk_size:
            # Flush current chunk first
            if current_chunk.strip():
                chunks.append(current_chunk.strip())
                current_chunk = ""
            
            # Word-boundary fallback for oversized blocks
            start = 0
            while start < len(block):
                end = start + chunk_size
                if end < len(block):
                    space_pos = block.rfind(" ", start, end)
                    if space_pos > start:
                        end = space_pos
                sub = block[start:end].strip()
                if sub:
                    chunks.append(sub)
                next_start = end - overlap
                start = max(next_start, start + 1)
            continue
        
        # Would adding this block exceed the budget?
        candidate = (current_chunk + "\n\n" + block).strip() if current_chunk else block
        if len(candidate) > chunk_size and current_chunk.strip():
            chunks.append(current_chunk.strip())
            current_chunk = block
        else:
            current_chunk = candidate
    
    # Flush the last chunk
    if current_chunk.strip():
        chunks.append(current_chunk.strip())
    
    # Apply overlap: prepend tail of previous chunk to each subsequent chunk
    if overlap > 0 and len(chunks) > 1:
        overlapped = [chunks[0]]
        for i in range(1, len(chunks)):
            prev = chunks[i - 1]
            # Take the last `overlap` characters, back up to word boundary
            if len(prev) > overlap:
                tail_start = len(prev) - overlap
                space_pos = prev.find(" ", tail_start)
                if space_pos != -1:
                    tail = prev[space_pos:].strip()
                else:
                    tail = prev[tail_start:].strip()
            else:
                tail = prev.strip()
            overlapped.append((tail + "\n\n" + chunks[i]).strip())
        chunks = overlapped
    
    return chunks

chunks = chunk_text_markdown(full_text, chunk_size=1000, overlap=200)

print(f"Total chunks created: {len(chunks)}")
print(f"Average chunk length: {sum(len(c) for c in chunks) / len(chunks):.0f} characters")

Total chunks created: 1205
Average chunk length: 973 characters


Now let's run the same inspection we ran against the naive chunks. But this time we are not just checking for broken words. Look at where the boundaries fall. Do headings stay with the content beneath them? Do paragraphs break at natural seams rather than arbitrary character counts? If a table appeared near a chunk boundary, did it survive intact?


The chunks will not be uniform in size. That is the point. We are trading uniformity for coherence.


In [12]:
# Same inspection — now check if boundaries are clean
print("--- End of Chunk  ---")
print(f"...{chunks[750][-80:]}")
print(f"\n--- Start of Chunk  ---")
print(f"{chunks[751][:80]}...")
print(f"\n{'=' * 60}")
print("Clean word boundaries now?")
print(f"{'=' * 60}")

--- End of Chunk  ---
...00 pounds.

<!-- image -->

## BASIC FANTASY RPG

## BASIC FANTASY RPG

## Pixie

--- Start of Chunk  ---
1,400 pounds, with a wingspan of 22 feet.  A light load for a pegasus is up to 4...

Clean word boundaries now?


### 4.3.6 Let's Try Searching Again

Before we move on, let's confirm that the content we actually care about survived the transition. Back in Section 2, we asked the models about the Thief's Open Locks ability and got generic answers. The correct rule is somewhere in this document. 

Let's make sure our chunking did not bury it.

In [13]:
# Verify our test case still shows up correctly
for i, chunk in enumerate(chunks):
    if "open locks" in chunk.lower():
        print(f"\n=== Chunk {i} (contains 'Open Locks') ===")
        print(chunk)
        print(f"Length: {len(chunk)} chars")
        print()


=== Chunk 75 (contains 'Open Locks') ===
fit; for instance, it's obviously harder to climb a wall slick with slime than one that is dry, so the GM might apply a penalty of 20% for the slimy wall.

## BASIC FANTASY RPG

## Thief Abilities

|   Thief Level |   Open Locks |   Remove Traps |   Pick Pockets |   Move Silently |   Climb Walls |   Hide |   Listen |
|---------------|--------------|----------------|----------------|-----------------|---------------|--------|----------|
|             1 |           25 |             20 |             30 |              25 |            80 |     10 |       30 |
|             2 |           30 |             25 |             35 |              30 |            81 |     15 |       34 |
|             3 |           35 |             30 |             40 |              35 |            82 |     20 |       38 |
|             4 |           40 |             35 |             45 |              40 |            83 |     25 |       42 |
|             5 |           45 |  

The good news: the rule is intact. Chunk 79 contains the complete Open Locks description in a clean paragraph, unbroken, with the sentences before and after it providing natural context. A retriever can find this. A model can answer from it. Compare that to what the naive chunker would have produced: the same text sliced at an arbitrary character boundary, potentially splitting "Open Locks" across two chunks where neither one carried the full rule.


Now look at `Chunk 75`. The Thief Abilities table is here, and the structural chunker kept a meaningful portion of it together: the heading, the column headers, and the first several rows all landed in the same chunk. But the table is large. It exceeds our chunk size on its own, so the bottom rows spilled into subsequent chunks. The fallback logic did its job: it split at a row boundary rather than in the middle of a cell, but the table is still fragmented.


This is the tradeoff we described. Structural chunking keeps most things intact. It does not guarantee that everything stays intact. A table that exceeds the chunk size will still be split. The difference is where and how. The naive chunker would have cut through the middle of a row. This version at least preserves row boundaries and keeps the header with the data.


For this lab, the result is good enough. The rule we need is retrievable as a complete, coherent paragraph. In a production system with dense tables, cross-referenced sections, or multi-part definitions, this is exactly where you would start looking at table-specific handling: extracting tables as standalone chunks regardless of size, or storing them separately with their own metadata. That is a conversation for after the lab.


What matters now is the difference between what we had and what we have. The naive chunker gave us broken words and arbitrary boundaries. The markdown-aware chunker gives us structural breaks and coherent paragraphs. The table problem is visible, understood, and addressable. That is progress.

### 4.3.7 Experiment: What Happens With Different Chunk Sizes?

We've been using 1000/200 as our baseline. That's a reasonable starting point — but it's not a universal answer. The table split we just saw is partly a chunk-size problem: a larger chunk might have kept the table intact, but at the cost of pulling in less relevant surrounding text on every retrieval.

The tradeoff is always the same. Smaller chunks mean more of them, more precise retrieval, but each chunk carries less context. Larger chunks mean fewer of them, more context per retrieval, but the retriever becomes less targeted.

There's no formula that resolves this. There's only awareness and, of course, the willingness to test.


In [14]:
for size in [200, 500, 1000, 2000]:
    test_chunks = chunk_text_markdown(full_text, chunk_size=size, overlap=int(size * 0.2))
    print(f"Chunk size: {size:>5}  |  Overlap: {int(size*0.2):>4}  |  Total chunks: {len(test_chunks):>4}")

Chunk size:   200  |  Overlap:   40  |  Total chunks: 7225
Chunk size:   500  |  Overlap:  100  |  Total chunks: 2723
Chunk size:  1000  |  Overlap:  200  |  Total chunks: 1205
Chunk size:  2000  |  Overlap:  400  |  Total chunks:  542


### 4.3.8 More resources on Chunking Strategies

If this section felt like we only scratched the surface, that’s because we did.
There isn’t one “correct” way to chunk documents for RAG systems. There are multiple strategies, each with tradeoffs depending on your domain, document structure, query patterns, and latency constraints. A full taxonomy of chunking approaches, and when to use each, is a deep dive of its own.

That’s outside the scope of this lab.

The goal here wasn’t to turn you into a chunking theorist. It was to recalibrate how seriously you take this step. I want attendees to walk away with a deep respect for how profoundly the chunking process impacts the quality of the end result. Retrieval quality, answer precision, user trust: all of it is shaped upstream by how you slice the data.

If you’d like to go deeper, here are a few solid starting points:
* “Five Levels of Chunking Strategies in RAG” by Anurag Mishra
  * https://medium.com/@anuragmishra_27746/five-levels-of-chunking-strategies-in-rag-notes-from-gregs-video-7b735895694d
* Text splitters documentation: Text splitters break large docs into smaller chunks that will be retrievable individually and fit within the model context window limit.
  * https://docs.langchain.com/oss/python/integrations/splitters 
* Technical blog posts from vector database vendors (Weaviate, Pinecone, etc.) that compare fixed-size, semantic, recursive, and structure-aware chunking approaches
* Conference talks and YouTube deep dives on “RAG architecture” and “context engineering,” which often include practical chunking heuristics and failure modes

As you explore, pay attention to the tradeoffs each approach optimizes for: coherence, recall, precision, latency, or cost. There are many ways to solve this problem.

The important thing is that you now see chunking for what it is: leverage not just plumbing.


## 4.4 Embeddings and the Vector Store

Now we turn our chunks into something searchable.

A RAG system relies on two separate models, and it's important not to confuse them. The embedding model converts text into numerical vectors — dense lists of numbers that capture semantic meaning. Chunks about similar topics produce similar vectors, even if they use different words. It's small, it's fast, and it doesn't generate text. It just maps language to numbers.

The language model is what we've already been using: `granite-3-2-8b-instruct`. Its job comes later: generating answers from whatever context we send it.

Between them sits the vector store. It holds all the chunk embeddings and provides fast similarity search. 

When a question arrives, the system embeds the question using the same embedding model, finds the chunks whose vectors are closest to the question's vector, and returns those chunks as context for the language model.

We're using ChromaDB here because it runs inside this notebook with no external infrastructure. The point isn't the tool: it's the pattern: embed, index, retrieve.


### 4.4.1 Load the Embedding Model


In [15]:
from sentence_transformers import SentenceTransformer
print("Loading Granite embedding model...")
embed_model = SentenceTransformer("ibm-granite/granite-embedding-30m-english")
print("✅ Embedding model loaded")

Loading Granite embedding model...


modules.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/683 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

✅ Embedding model loaded


/opt/app-root/lib64/python3.12/site-packages/torch/cuda/__init__.py:789: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


When the cell completes, you should see a confirmation that the model loaded successfully.

### 4.4.2 Embed All Chunks
Now we take every chunk from our markdown-aware splitting and embed it into the vector store.  Each chunk gets converted into a vector, tagged with its index and source file, and stored in a ChromaDB collection. 

This is the step where text becomes searchable by meaning rather than by keyword. It may take a minute or two depending on the environment; that's normal. 

Over a thousand chunks are being embedded in sequence.


In [16]:
print(f"Embedding {len(chunks)} chunks...")
embeddings = embed_model.encode(chunks, show_progress_bar=False, batch_size=64)
print(f"✅ Embeddings complete — shape: {embeddings.shape}")

Embedding 1205 chunks...
✅ Embeddings complete — shape: (1205, 384)


When the cell finishes, you should see something like:

`Embedding 1205 chunks...`

`✅ Embeddings complete — shape: (1205, 384)`

That shape tells you exactly what happened. The first number `1205` is how many chunks were embedded, one vector per chunk. The second number `384` is the dimensionality of each vector. 

Every chunk in our document is now represented as a list of `384` numbers. Those numbers encode semantic meaning: chunks about similar topics will have similar vectors, even if they don't share the same words. That's what makes similarity search possible and it's what separates retrieval from keyword matching.

This is the bridge between text and math. From here, "find relevant context" becomes "find the nearest vectors."

In [17]:
!pip install pysqlite3-binary -q

With the updated SQLite binding in place, we also need to swap it in before importing ChromaDB. The three lines at the top of the next cell handle that. The telemetry warnings in the output are harmless and can be ignored.

In [18]:
embeddings = embed_model.encode(chunks, show_progress_bar=False, batch_size=64)

__import__('pysqlite3')
import sys
sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')


import chromadb
client = chromadb.PersistentClient(path="../chroma_db")

collection = client.get_or_create_collection(
    name="rpg_rules",
    metadata={"hnsw:space": "cosine"}
)

# Pass pre-computed embeddings
collection.add(
    documents=chunks,
    embeddings=embeddings.tolist(),
    ids=[f"chunk_{i}" for i in range(len(chunks))]
)

print(f"✅ Loaded {collection.count()} chunks into ChromaDB")

✅ Loaded 1205 chunks into ChromaDB


That's the retrieval infrastructure in place. Our document went from a PDF to structured Markdown to word-boundary-respecting chunks to searchable vectors in a store we can query. No external services, no complex infrastructure. Just a local embedding model and an in-memory database.
Now let's see if it can actually find the right answer.

## 4.5 Retrieval: Can We Find the Right Paragraph?

We've built the infrastructure. Documents were ingested, converted to Markdown, chunked with word-boundary awareness, embedded, and loaded into a vector store. That's a lot of plumbing. Now we find out if it actually works.

The test is simple. We ask the same question that stumped the models back in Section 2, and we see whether the retriever can find the chunk that contains the answer. Not generate an answer. Not reason over it. Just find it.

If the right paragraph comes back, we know the system can locate relevant content. If it doesn't, no amount of prompt engineering or model swapping will fix the problem. Retrieval is the gate. Everything downstream depends on what passes through it.


In [19]:
question = "What happens if a Thief fails an Open Locks attempt?"

# Embed the question with the same model used for chunks
question_embedding = embed_model.encode([question]).tolist()

# Retrieve the top 3 most relevant chunks
results = collection.query(
    query_embeddings=question_embedding,
    n_results=3
)

print(f"Question: {question}\n")
for i, (doc, distance) in enumerate(zip(results['documents'][0], results['distances'][0])):
    similarity = 1 - distance
    print(f"--- Retrieved Chunk {i+1} (similarity: {similarity:.3f}) ---")
    print(doc)
    print()

Question: What happens if a Thief fails an Open Locks attempt?

--- Retrieved Chunk 1 (similarity: 0.804) ---
82 |             98 |              91 |            98 |     72 |       92 |
|            20 |           88 |             83 |             99 |              93 |            99 |     73 |       95 |

The   numbers   above   are   percentages;   instructions   for making these rolls are in Using the Dice on page 2.

Open Locks allows the Thief to unlock a lock without a proper key.   It may only be tried once per lock.   If the attempt fails, the Thief must wait until they have gained another level of experience before trying again.

Remove Traps is generally rolled twice: first to detect the trap, and second to disarm it.   The GM will make these rolls as the player won't know for sure if the character is successful or not until someone actually tests the trapped (or suspected) area.

Pick Pockets allows the Thief to lift the wallet, cut the purse, etc. of a victim without being 

`Chunk 1` has the answer, right there in the middle: the Thief gets one attempt per lock, and if it fails, they wait until the next level. That's exactly the rule from page 9 of the source document. The retriever found it across 1,143 chunks with a similarity score of 0.817.

Notice what came along for the ride. The chunk also contains the tail end of a table, the Open Locks rule, the Remove Traps rule, and the beginning of Pick Pockets. That's the tradeoff we discussed: at a 1000-character chunk size, you get enough context for the answer to make sense, but you also pull in neighboring content. The model will have to focus on the relevant part. That's its job.

`Chunks 2 and 3` are related but not directly relevant. They describe other Thief abilities and general dungeon mechanics. The retriever ranked them lower, and rightly so. In a production system, you'd tune n_results and possibly add a similarity threshold to control how much context reaches the model. For now, the important thing is that the right paragraph was the top result.

Compare this to Section 2, where the models confidently referenced Dungeons & Dragons. The model hasn't changed. The prompt hasn't changed. The only difference is that the system now has the right paragraph to work with.

Retrieval found the right paragraph. 

But retrieval doesn't answer questions. It just hands context to something that can. The next step is to take these retrieved chunks, pass them to `granite-3-2-8b-instruct` as context, and ask the same question again. This time, the model won't be guessing from its training data. It will be reading from the customer's document.

This is the moment where RAG either proves its value or exposes its limits.

## 4.6 The Complete RAG Pipeline: Retrieve, Then Generate
Now we connect it all. The function below retrieves the most relevant chunks for a question, constructs a prompt that includes system instructions, the retrieved context, and the question itself, then sends it to the language model for a grounded answer.

Notice one critical detail in the prompt construction: we tell the model explicitly to answer only from the provided context. That's not a suggestion to the model. It's the mechanism that produces grounded answers, the exact quality the customer needs from Section 1.4. Without that instruction, the model will happily fill gaps with its training data, and you're back to confident answers about the wrong game.

The following cell defines the `rag_query` function.


In [26]:
def rag_query(question, collection, embed_model, api_key, base_url,
              model="granite-3-2-8b-instruct", n_results=3, max_tokens=300):
    # Step 1: Embed the question and retrieve relevant chunks
    q_embedding = embed_model.encode([question]).tolist()
    results = collection.query(
        query_embeddings=q_embedding,
        n_results=n_results
    )
    retrieved_chunks = results['documents'][0]
    
    # Step 2: Build the grounded prompt
    context = "\n\n---\n\n".join(retrieved_chunks)
    
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant. Answer the question using ONLY "
                "the provided context. If the context does not contain enough "
                "information to answer, say so explicitly. Do not make up information."
            )
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {question}"
        }
    ]
    
    # Step 3: Generate the answer
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    data = {
        "model": model,
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": 0
    }
    
    url = f"{base_url}/chat/completions"
    response = requests.post(url, headers=headers, json=data)
    response.raise_for_status()
    
    result = response.json()
    answer = result["choices"][0]["message"]["content"]
    
    # Token usage
    usage = result.get("usage", {})
    print("------------------------------------")
    print(f"Prompt tokens:     {usage.get('prompt_tokens')}")
    print(f"Completion tokens: {usage.get('completion_tokens')}")
    print(f"Total tokens:      {usage.get('total_tokens')}")
    print("------------------------------------")
    
    return {
        "question": question,
        "answer": answer,
        "retrieved_chunks": retrieved_chunks,
        "model": model
    }

### 4.6.1 Test It — Same Model, Same Question, Different Result

This is the comparison that matters. We're going to run the same question twice: once without any context, the way the models saw it in Section 2, and once through our RAG pipeline with retrieved chunks from the customer's document. Same model, same question, same API. The only variable is whether the model has the right paragraph to work with.

If the answer improves, we'll know exactly what caused it. And if it doesn't, we'll know exactly where to look.

Run the following code.

In [27]:
result = rag_query(
    question="What happens if a Thief fails an Open Locks attempt?",
    collection=collection,
    embed_model=embed_model,
    api_key=key,
    base_url=endpoint_base
)

print(f"Question: {result['question']}\n")
print(f"RAG Answer:\n{result['answer']}\n")
print(f"Model used: {result['model']}")
print(f"Context chunks used: {len(result['retrieved_chunks'])}")

------------------------------------
Prompt tokens:     911
Completion tokens: 44
Total tokens:      955
------------------------------------
Question: What happens if a Thief fails an Open Locks attempt?

RAG Answer:
If a Thief fails an Open Locks attempt, they won't be able to unlock the lock without a proper key. Furthermore, they must wait until they have gained another level of experience before trying again.

Model used: granite-3-2-8b-instruct
Context chunks used: 3


Compare that to Section 2, where the same model responded with generic references to Dungeons & Dragons. The answer is now grounded, specific, and correct: one attempt per lock, wait until the next level. That's straight from page 9 of the customer's rulebook.

Look at the token usage. Back in Section 4.1, we tried stuffing the entire document into context and hit 205,000 tokens before the server rejected it. RAG just answered the same question with less than half a percent of that cost. 

The model didn't need the whole book. 

It needed three paragraphs.


## 4.7 The Comparison: Baseline vs. RAG

This is why we established a baseline in Section 2. Without a "before," you can't prove "after." Without a baseline, every improvement is anecdotal. With one, it's evidence.
We'll run the same questions through both paths — the model alone and the model with retrieved context — and compare the results side by side. The model hasn't changed. The questions haven't changed. The only difference is whether the system gives the model something real to work with.

>Facilitator note: This is the key teaching moment. Pause here. Let participants read both answers. Ask: "Which answer would you trust in front of your legal team?"

In [28]:
questions = [
    "What happens if a Thief fails an Open Locks attempt?",
    "Why can't Elves roll higher than a d6 for hit points?",
    "What is the maximum number of retainers a character can have?",
    "How does turning undead work for Clerics?",
]

headers = {
    "Authorization": f"Bearer {key}",
    "Content-Type": "application/json"
}

print("=" * 80)
print("BASELINE vs RAG COMPARISON")
print("=" * 80)

for q in questions:
    print(f"\n{'─' * 80}")
    print(f"QUESTION: {q}")
    print(f"{'─' * 80}")
    
    # Baseline: no context
    baseline_data = {
        "model": "granite-3-2-8b-instruct",
        "messages": [{"role": "user", "content": q}],
        "max_tokens": 200,
        "temperature": 0
    }
    try:
        baseline_resp = requests.post(url_chat, headers=headers, json=baseline_data)
        baseline_answer = baseline_resp.json()["choices"][0]["message"]["content"]
    except Exception as e:
        baseline_answer = f"[Error: {e}]"
    
    # RAG: with retrieved context
    try:
        rag_result = rag_query(q, collection, embed_model, key, endpoint_base)
        rag_answer = rag_result["answer"]
    except Exception as e:
        rag_answer = f"[Error: {e}]"
    
    print(f"\n📭 BASELINE (no context):\n{baseline_answer}")
    print(f"\n📬 RAG (with retrieval):\n{rag_answer}")

print(f"\n{'=' * 80}")
print("Review: Which answers improved? Which didn't? Why?")
print(f"{'=' * 80}")

BASELINE vs RAG COMPARISON

────────────────────────────────────────────────────────────────────────────────
QUESTION: What happens if a Thief fails an Open Locks attempt?
────────────────────────────────────────────────────────────────────────────────
------------------------------------
Prompt tokens:     911
Completion tokens: 44
Total tokens:      955
------------------------------------

📭 BASELINE (no context):
If a Thief fails an Open Locks attempt, they do not make any progress towards opening the lock. The Thief must wait until their next attempt to try again. If the Thief fails the roll by 4 or more, they may trigger a trap or alert nearby enemies.

📬 RAG (with retrieval):
If a Thief fails an Open Locks attempt, they won't be able to unlock the lock without a proper key. Furthermore, they must wait until they have gained another level of experience before trying again.

────────────────────────────────────────────────────────────────────────────────
QUESTION: Why can't Elves 

Read the pairs. The pattern is consistent across all four questions.

The baseline answers sound plausible. They're well-structured, confident, and wrong. The first invents a `"fail by 4 or more"` trap trigger that doesn't exist in the rules. The second fabricates a lore-based rationale for Elf hit dice instead of citing the actual restriction. The third confidently states a limit of 3 retainers from D&D 5th Edition — the wrong game entirely. The fourth describes a Wisdom saving throw mechanic that belongs to a completely different ruleset.

The RAG answers are shorter, less polished, and correct. Open Locks: one try per lock, wait a level. Elf hit dice: restricted to d6, explicitly stated. Retainers: up to 4, adjusted by Charisma. Turning undead: roll d20, consult the table, follow the specific procedure from the document.

Notice the token counts. Every RAG call landed under 1,200 total tokens. We're getting accurate, grounded answers from a fraction of the document — not because the model got smarter, but because it received the right three paragraphs instead of nothing at all.

This is the core argument for RAG. The model didn't change. The questions didn't change. The only thing that changed is what the model could see when it answered.

## 4.8 Making Answers Defensible: Show the Evidence

A correct answer isn't enough. The customer needs to **explain why the system said what it said.**

This is one of the key advantages of RAG over fine-tuning: you can always point to the exact source chunks that informed the answer. This makes the system **auditable.**

> "Would you trust this in front of your legal / compliance / engineering team?"

If you can show the source, the answer to that question changes.


In [29]:
question = "What happens if a Thief fails an Open Locks attempt?"
result = rag_query(question, collection, embed_model, key, endpoint_base)

print(f"Question: {result['question']}\n")
print(f"Answer:\n{result['answer']}\n")
print("=" * 60)
print("EVIDENCE — Retrieved chunks that informed this answer:")
print("=" * 60)
for i, chunk in enumerate(result['retrieved_chunks']):
    print(f"\n--- Source Chunk {i+1} ---")
    print(chunk)

------------------------------------
Prompt tokens:     911
Completion tokens: 44
Total tokens:      955
------------------------------------
Question: What happens if a Thief fails an Open Locks attempt?

Answer:
If a Thief fails an Open Locks attempt, they won't be able to unlock the lock without a proper key. Furthermore, they must wait until they have gained another level of experience before trying again.

EVIDENCE — Retrieved chunks that informed this answer:

--- Source Chunk 1 ---
82 |             98 |              91 |            98 |     72 |       92 |
|            20 |           88 |             83 |             99 |              93 |            99 |     73 |       95 |

The   numbers   above   are   percentages;   instructions   for making these rolls are in Using the Dice on page 2.

Open Locks allows the Thief to unlock a lock without a proper key.   It may only be tried once per lock.   If the attempt fails, the Thief must wait until they have gained another level of ex

Look at Source Chunk 1. The sentence is right there: "It may only be tried once per lock. If the attempt fails, the Thief must wait until they have gained another level of experience before trying again." The model's answer is a direct restatement of that rule. You can trace every claim in the answer back to a specific sentence in the evidence.

That's the audit trail. Not a footnote, not a vague attribution to "the rulebook." The actual text the model was reading when it generated the answer. A reviewer who has never seen this system before can look at this output and verify the answer in under a minute.

Chunks 2 and 3 are noise in this case. They describe related Thief abilities but don't contribute to the answer. In a production system, you might tighten retrieval to reduce that noise, or add logic that highlights which chunks the model actually drew from. But even with three chunks and some irrelevant context, the system produced a correct, verifiable answer. That's a defensible result.

## 4.9 Decision Point: Is This Good Enough?
We got the right answer. That's real progress, and it's worth acknowledging. The same model that confidently cited the wrong game in Section 2 just produced a grounded, verifiable answer from the customer's actual document.

But one right answer is not confidence. It's a data point. An anecdote. A demo.

Before we reach for improvements — better chunking, different models, more documents — there's a more fundamental question in front of us. How do we know when this system is working, and when it isn't?

So far we tested a single question. We knew the answer because we manually checked the source document. That approach works in a lab with one question and one PDF. It does not work in the real world, where a customer may have hundreds of questions across thousands of pages, and no one is reading every response to verify it.

The next step is not model changes. It's not better chunking. It's not adding more documents.

It's evaluation.

Without evaluation, every improvement is intuition disguised as engineering. Improve the chunking strategy? Maybe it helps. How would you know? Swap the embedding model? Compared to what? Add more source documents? Did quality improve, decline, or just change in ways nobody measured?

Evaluation is what turns a demo into a system you can reason about. We've proven the pipeline can produce a good answer. Now we need to understand how often it does — and what to do when it doesn't.


# Now on to Section 5!